In [ ]:
import geopandas as gpd

# Load tracts and ZCTAs
tracts = gpd.read_file("tl_2020_06_tract.shp").to_crs(3310)
zctas = gpd.read_file("tl_2020_us_zcta510.shp").to_crs(3310)

# Spatial overlay to compute intersections
intersections = gpd.overlay(tracts, zctas, how="intersection")

# Compute area of intersecting region
intersections["area"] = intersections.geometry.area

# For each tract, keep the ZCTA with largest intersected area
idx = intersections.groupby("GEOID")["area"].idxmax()
tract_to_zcta = intersections.loc[idx, ["GEOID", "ZCTA5CE10", "area"]]
tract_to_zcta.rename(columns={"ZCTA5CE10": "zip"}, inplace=True)

# Save CSV for merging later
tract_to_zcta.to_csv("tract_zip_crosswalk.csv", index=False)


In [ ]:
age = pd.read_csv("age_by_tract.csv")
xwalk = pd.read_csv("tract_zip_crosswalk.csv")

age_zip = age.merge(xwalk, left_on="tract", right_on="GEOID")


In [ ]:
zip_labels = pd.read_csv("zip_underserved.csv")
full = age_zip.merge(zip_labels, on="zip")
